# Project 3 — Capacity and Resource Allocation Optimisation
## 01 · Formulation and greedy baseline

### Analytical question
Given a week's leads, provider capacity and match quality, how should notifications be allocated?

### Evidence so far
Project 1 showed that lead-by-lead ranking leaves leads unfilled once capacity tightens.

### Why this approach?
Formulate as a MILP (see `src/formulation.py` docstring): binary notify decisions, per-lead and per-provider limits, soft penalties for unfilled leads and for providers who receive nothing. The greedy baseline is the current behaviour formalised.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
os.chdir(os.path.abspath('../..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 160)
%matplotlib inline
from project3_allocation.src import instance, analysis
inst = instance.build('data/synthetic', '2026-01-19', 'outputs/project1/model_logreg.joblib')
print(len(inst.leads), 'leads', inst.capacity.sum(), 'capacity', len(inst.pairs), 'feasible pairs')
analysis.compare_solvers(inst).set_index('solver').T

23:26:51 p1.data INFO: providers: 224 -> 220 after de-duplication


23:26:51 p1.data INFO: leads: 3000 -> 2936 after dropping unknown zone


23:26:51 p1.data INFO: offers: 14648 -> 14645 after referential-integrity filter


23:26:52 p3.instance INFO: week 2026-01-19: 164 leads, 220 providers, 5387 feasible pairs


164 leads 635 capacity 5387 feasible pairs


solver,greedy,milp
quality,250.145542,278.626339
unfilled_penalty,7.0,0.0
fairness_penalty,5.1,0.0
distance_penalty,8.804415,6.153436
objective,229.241128,272.472904
leads_unfilled,4,0
leads_below_min,6,0
providers_with_zero,17,0
notifications,462,487
mean_q,0.541441,0.572128


### Result / Interpretation
The optimiser lifts expected purchases by ~11% over greedy, fills every lead and leaves no credit-holding provider without a lead — all on the same capacity. Greedy is myopic: early leads consume providers a later priority lead needed.

### Decision
MILP with CBC (D2); weekly batch with a re-solve when new leads arrive.